# Day 6 - Error Analysis

Systematically buckets every false negative (FN) and false positive (FP)
from both approaches by entity type, draws a stratified sample for manual
annotation, and produces the error-distribution figure used in the report.

**Prerequisite outputs (read-only)**
| File | Produced by |
|---|---|
| `predictions/encoder/deberta_s42_predictions.jsonl` | Day 5 - `05_evaluate_all.py` |
| `predictions/llm/raw_outputs.jsonl` | Day 4 - `04_llm_inference.py` |
| `reports/results.json` | Day 5 - `05_evaluate_all.py` |

**Outputs written here**
| Artefact | Location |
|---|---|
| Error bucket counts | `results/day6/error_buckets.json` |
| Manual annotation sheets | `errors/manual_A.tsv`, `errors/manual_B.tsv` |
| Error distribution figure | `reports/figures/fig1_error_analysis.png` |

## 1  Setup

In [1]:
import json
import pathlib
import sys

# Resolve project root so the src package is importable from the notebook
ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pii_masking.day6_error_analysis import (
    BucketResult,
    build_tsv_rows,
    collect_error_instances,
    compute_buckets,
    load_predictions,
    markdown_table_str,
    plot_error_distribution,
    sample_distribution_str,
    save_tsv,
    stratified_sample,
    summary_table_str,
)

print("Imports OK  |  ROOT:", ROOT)

Imports OK  |  ROOT: E:\Projects\pii_masking


In [2]:
# PathsAPPROACH_A_FILE    = ROOT / "predictions" / "encoder" / "deberta_s42_predictions.jsonl"
APPROACH_B_FILE    = ROOT / "predictions" / "llm"     / "raw_outputs.jsonl"
RESULTS_JSON       = ROOT / "reports" / "results.json"
RESULTS_DAY6_DIR   = ROOT / "results"  / "day6"
ERROR_BUCKETS_JSON = RESULTS_DAY6_DIR / "error_buckets.json"
ERRORS_DIR         = ROOT / "errors"
FIGURES_DIR        = ROOT / "reports" / "figures"
FIGURE_PNG         = FIGURES_DIR / "fig1_error_analysis.png"

# Create output directories up-front
for d in (RESULTS_DAY6_DIR, ERRORS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Output directories ready.")

Output directories ready.


## 2  Load predictions

In [3]:
records_a, records_b = load_predictions(APPROACH_A_FILE, APPROACH_B_FILE)

print(f"Approach A (DeBERTa s42) : {len(records_a):,} sentences")
print(f"Approach B (LLaMA)       : {len(records_b):,} sentences")

# Sanity-check: first record from each approach
print("\n--- Approach A sample ---")
r = records_a[0]
print("tokens   :", r["tokens"][:8], "...")
print("gold_tags:", r["gold_tags"][:8], "...")
print("pred_tags:", r["pred_tags"][:8], "...")

print("\n--- Approach B sample ---")
r = records_b[0]
print("tokens   :", r["tokens"][:8], "...")
print("gold_tags:", r["gold_tags"][:8], "...")
print("pred_tags:", r["pred_tags"][:8], "...")

Approach A (DeBERTa s42) : 3,650 sentences
Approach B (LLaMA)       : 3,650 sentences

--- Approach A sample ---
tokens   : ['This', 'Is', 'What', 'the', 'Truth', 'Feels', 'Like', '"'] ...
gold_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...
pred_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...

--- Approach B sample ---
tokens   : ['This', 'Is', 'What', 'the', 'Truth', 'Feels', 'Like', '"'] ...
gold_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...
pred_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...


## 3  Error bucket counts

Buckets are defined by the cross-product of direction (FN = missed entity, FP = spurious entity)
and entity type (PER, EMAIL).  Each span is compared at **exact** boundary level - both endpoints
and the label must match the gold annotation.

In [4]:
buckets_a = compute_buckets(records_a)
buckets_b = compute_buckets(records_b)

print(summary_table_str(buckets_a, buckets_b))

Bucket        Approach A      A %  Approach B      B %
------------------------------------------------------
FN_PER                88    40.2%       1,374     9.4%
FN_EMAIL               1     0.5%         707     4.8%
FP_PER               130    59.4%       9,261    63.3%
FP_EMAIL               0     0.0%       3,289    22.5%
------------------------------------------------------
TOTAL                219               14,631


In [5]:
# Save to disk
payload = {
    "approach_A": buckets_a.to_dict(),
    "approach_B": buckets_b.to_dict(),
}
ERROR_BUCKETS_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Saved -> {ERROR_BUCKETS_JSON}")

Saved -> E:\Projects\pii_masking\results\day6\error_buckets.json


## 4  Error distribution figure

Grouped bar chart with Approach A (DeBERTa) in blue and Approach B (LLaMA) in orange.
Note the log-scale difference in total errors: DeBERTa produces **219** errors vs
LLaMA's **14 631** - two orders of magnitude more noise.

In [6]:
import matplotlib
matplotlib.use("Agg")          # headless; swap to "inline" if running interactively
import matplotlib.pyplot as plt

fig = plot_error_distribution(buckets_a, buckets_b, FIGURE_PNG, dpi=150)

# Re-render inline in the notebook (Agg backend writes the file; re-open to display)
fig2, ax2 = plt.subplots(figsize=(8, 5))
img = plt.imread(str(FIGURE_PNG))
ax2.imshow(img)
ax2.axis("off")
plt.tight_layout()
plt.show()
print(f"Figure saved -> {FIGURE_PNG}")

Figure saved -> E:\Projects\pii_masking\reports\figures\fig1_error_analysis.png


C:\TEMP\ipykernel_28844\527768489.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
print(markdown_table_str(buckets_a, buckets_b))

| Bucket   | Approach A | % A    | Approach B |  % B   |
|----------|------------|--------|------------|--------|
| FN_PER   |         88 |  40.2% |      1,374 |   9.4% |
| FN_EMAIL |          1 |   0.5% |        707 |   4.8% |
| FP_PER   |        130 |  59.4% |      9,261 |  63.3% |
| FP_EMAIL |          0 |   0.0% |      3,289 |  22.5% |
| **TOTAL** |        219 |        |     14,631 |        |
